# HargaWatch Surabaya: Analisis Fitur & Eksplorasi Deret Waktu (EDA Khusus)

Notebook ini secara khusus dibuat untuk memvisualisasikan dan membuktikan keputusan-keputusan strategis dalam *Preprocessing* dan *Feature Engineering* sebelum data masuk ke dalam model *Machine Learning*.

## Daftar Isi:
1. **Bukti Preprocessing**: Kontinuitas Data & *Forward-Fill*.
2. **Disparitas Pasar**: Peran Pasar Induk Keputran.
3. **Efek Kalender**: Dampak Pra-Ramadan terhadap harga.
4. **Korelasi Cuaca**: Fungsi Korelasi Silang (CCF) Hujan vs Cabai.
5. **Pemetaan Volatilitas**: Komoditas stabil vs bergejolak.
6. **Hasil Akhir Feature Engineering**: Matriks siap latih.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup plotting
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
df_harga = pd.read_csv(BASE_DIR / 'data/processed/fact_harga_pasar.csv', parse_dates=['tanggal'])
df_kalender = pd.read_csv(BASE_DIR / 'data/processed/dim_kalender.csv', parse_dates=['tanggal'])
df_cuaca = pd.read_csv(BASE_DIR / 'data/external/cuaca/cuaca_surabaya.csv', parse_dates=['tanggal'])
df_pasar = pd.read_csv(BASE_DIR / 'data/processed/dim_pasar.csv')
df_komoditas = pd.read_csv(BASE_DIR / 'data/processed/dim_komoditas.csv')

# Gabungkan untuk kemudahan plotting
df = df_harga.merge(df_kalender, on='tanggal', how='left')
df = df.merge(df_pasar[['pasar_id', 'nama_pasar']], on='pasar_id', how='left')
print('Data siap dieksplorasi!')


## 1. Metode Preprocessing: Kontinuitas Data Tanpa Lookahead Bias
Dalam deret waktu, hari libur sering tidak ada pencatatan harga. Kita **dilarang menggunakan interpolasi linier** (menarik garis lurus antara hari Jumat dan Senin) karena interpolasi linear membutuhkan informasi hari Senin saat memprediksi hari Sabtu (Kebocoran Masa Depan / *Lookahead Bias*).

**Metode yang kita gunakan:** *Forward-Fill* murni. Harga hari Jumat dipertahankan untuk hari Sabtu dan Minggu.

In [ ]:
# Contoh pada Beras Premium di Pasar Wonokromo (Bulan tertentu)
sample = df[(df['komoditas_id'] == 2) & (df['pasar_id'] == 2) & 
            (df['tanggal'] >= '2025-01-01') & (df['tanggal'] <= '2025-01-31')].sort_values('tanggal')

plt.figure(figsize=(14, 5))
plt.plot(sample['tanggal'], sample['harga_imputasi'], marker='o', linestyle='-', color='#16A34A', label='Harga Imputasi (Forward-Fill)')
plt.scatter(sample[sample['is_imputed'] == 1]['tanggal'], 
            sample[sample['is_imputed'] == 1]['harga_imputasi'], 
            color='#DC2626', s=100, zorder=5, label='Hari Libur/Kosong (Hasil Imputasi)')
plt.title('Kontinuitas Data: Pengisian Kekosongan Tanpa Lookahead Bias', fontweight='bold')
plt.ylabel('Harga (Rp)')
plt.legend()
plt.show()


## 2. Disparitas Antar-Pasar (Standardisasi)
Pasar Keputran bertindak sebagai **Pasar Induk (Grosir)**. Harganya secara struktural lebih rendah dari pasar eceran. Inilah alasan mengapa pasar_id dimasukkan sebagai fitur model Global kita, agar ML paham konteks harga grosir vs eceran.

In [ ]:
cabe_all = df[(df['komoditas_id'] == 50) & (df['tanggal'] >= '2025-01-01')]
plt.figure(figsize=(14, 6))
sns.lineplot(data=cabe_all, x='tanggal', y='harga_imputasi', hue='nama_pasar', alpha=0.7)
plt.title('Disparitas Pasar: Grosir (Keputran) vs Eceran (Cabe Rawit Merah)', fontweight='bold')
plt.ylabel('Harga (Rp/Kg)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 3. Efek Musiman Kalender (Ramadan & Pra-Ramadan)
Kita mengekstrak fitur is_ramadan dan is_pra_ramadan karena harga pangan tertentu (seperti Cabai dan Daging) memiliki lonjakan tajam 14 hari sebelum puasa (Pra-Ramadan) akibat naiknya permintaan.

In [ ]:
# Buat kolom fase
conditions = [
    cabe_all['is_pra_ramadan'] == 1,
    cabe_all['is_ramadan'] == 1
]
choices = ['Pra-Ramadan', 'Ramadan']
cabe_all['Fase Kalender'] = np.select(conditions, choices, default='Normal')

plt.figure(figsize=(10, 6))
sns.boxplot(data=cabe_all, x='Fase Kalender', y='harga_imputasi', order=['Normal', 'Pra-Ramadan', 'Ramadan'], palette='Set2')
plt.title('Distribusi Harga Cabe Rawit Merah Berdasarkan Fase Ramadan', fontweight='bold')
plt.ylabel('Harga (Rp/Kg)')
plt.show()


## 4. Korelasi Cuaca (Fungsi Korelasi Silang / CCF)
Tanaman hortikultura sangat terpengaruh cuaca. Namun, dampaknya tidak terjadi hari ini juga (hujan hari ini tidak langsung membuat harga cabai naik hari ini). Kita mencari jeda waktu (Lag) yang paling berpengaruh menggunakan CCF.

In [ ]:
# Gabungkan harga Cabe (median kota) dengan curah hujan
cabe_median = df[df['komoditas_id'] == 50].groupby('tanggal')['harga_imputasi'].median().reset_index()
cabe_cuaca = cabe_median.merge(df_cuaca, on='tanggal', how='inner')

# Hitung korelasi silang (Lag 0 hingga 40 hari ke belakang)
lags = np.arange(0, 41)
corrs = [cabe_cuaca['harga_imputasi'].corr(cabe_cuaca['curah_hujan_mm'].shift(lag)) for lag in lags]

plt.figure(figsize=(12, 5))
plt.bar(lags, corrs, color='#3B82F6')
plt.axvline(28, color='red', linestyle='--', label='Lag 28 Hari (4 Minggu)')
plt.title('Korelasi Silang (CCF): Curah Hujan Historis vs Harga Cabai Rawit Saat Ini', fontweight='bold')
plt.xlabel('Lag Hujan (N Hari Sebelumnya)')
plt.ylabel('Korelasi Pearson')
plt.legend()
plt.show()


## 5. Klastering Volatilitas (Feature Engineering: Rolling Std)
Setiap komoditas memiliki volatilitas berbeda. Oleh karena itu, di eatures.py kita menggunakan **Rolling Standard Deviation (14 hari)** dan **Koefisien Variasi** sebagai fitur utama untuk mesin *Early Warning*.

In [ ]:
# Hitung volatilitas (Koefisien Variasi rata-rata per komoditas)
df_vol = df.groupby(['komoditas_id', 'pasar_id'])['harga_imputasi'].apply(lambda x: x.std() / x.mean()).reset_index(name='cv')
df_vol_agg = df_vol.groupby('komoditas_id')['cv'].mean().reset_index()
df_vol_agg = df_vol_agg.merge(df_komoditas[['komoditas_id', 'komoditas']], on='komoditas_id')
df_vol_agg = df_vol_agg.sort_values('cv', ascending=False).head(15)

plt.figure(figsize=(12, 7))
sns.barplot(data=df_vol_agg, x='cv', y='komoditas', palette='Reds_r')
plt.title('Tingkat Volatilitas Komoditas (Koefisien Variasi Historis)', fontweight='bold')
plt.xlabel('Volatilitas (Semakin besar semakin bergejolak)')
plt.ylabel('')
plt.show()


## 6. Hasil Akhir Feature Engineering
Berdasarkan semua temuan di atas, skrip scripts/ml/features.py mentransformasi data mentah menjadi bentuk matriks tabular yang siap disuapkan ke *LightGBM*. Fitur utamanya meliputi:
*   **Lags Autoregresif:** price_lag_1, price_lag_7, dll.
*   **Rolling Statistics:** 
olling_mean_7d, 
olling_std_14d.
*   **Kalender Target:** 	arget_is_ramadan dll (Fitur di titik masa depan +7$).
*   **Sinyal Cuaca Historis:** 
ain_sum_14d, 
ain_lag_28d.


In [ ]:
import sys
sys.path.insert(0, str(BASE_DIR))
from scripts.ml.features import build_supervised_dataset

# Menghasilkan dataset latih untuk horizon 7 hari ke depan
df_train = build_supervised_dataset(horizon=7, komoditas_id=50, include_weather=True)

print('=== SAMPEL MATRIKS FITUR (SIAP LATIH) ===')
cols_to_show = ['tanggal', 'nama_pasar', 'price_current', 'price_lag_7', 'rolling_std_14d', 
                'target_is_pra_ramadan', 'rain_lag_28d', 'target_price']
df_display = df_train.merge(df_pasar[['pasar_id', 'nama_pasar']], on='pasar_id', how='left')
display(df_display[cols_to_show].tail())
